In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("schema_name", "", "schema_name")
dbutils.widgets.text("table_name", "", "table_name")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
table_name = dbutils.widgets.get("table_name")

In [0]:
query = f"""
SELECT
    r.rule_name AS name,
    c.column_name AS column,
    r.rule_function AS function,
    m.criticality AS criticality,
    m.arguments AS arguments,
    r.rule_type,
    m.is_active
FROM {catalog_name}.{schema_name}.dqx_rule_mappings m
JOIN {catalog_name}.{schema_name}.dqx_columns c ON (m.table_id = c.table_id AND m.column_name = c.column_name)
JOIN {catalog_name}.{schema_name}.dqx_rule_definitions r ON m.rule_id = r.rule_id
JOIN {catalog_name}.{schema_name}.dqx_tables t ON m.table_id = t.table_id
WHERE t.table_name = '{table_name}'
"""

dqx_mapped_config_df = spark.sql(query).filter("is_active=true")
display(dqx_mapped_config_df)

if dqx_mapped_config_df.isEmpty():
    raise Exception("No active rules found for the table")

## Apply check

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from databricks.sdk import WorkspaceClient

from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule,DQDatasetRule
from databricks.labs.dqx import check_funcs
from databricks.labs.dqx.config import TableChecksStorageConfig
from databricks.labs.dqx.config import InputConfig, OutputConfig

## check by rules

In [0]:
checks_config = []
for row in dqx_mapped_config_df.collect():
    import json
    args = {}
    if row['arguments']:
        for k, v in row['arguments'].items():
            try:
                args[k] = json.loads(v)
            except Exception:
                args[k] = v
    
    if row['rule_type'] == 'row_level':
        checks_config.append(
            DQRowRule(
            name=row['name'],
            criticality=row['criticality'],
            check_func=getattr(check_funcs, row['function']),
            column=row['column']),
            **args
        )
    elif row['rule_type'] == 'column_level':
        checks_config.append(
            DQDatasetRule(
            name=row['name'],
            criticality=row['criticality'],
            check_func=getattr(check_funcs, row['function']),
            columns=[row['column']]),
            **args
        )

In [0]:
for i in (checks_config):
    print(i)

In [0]:
table_df = spark.table(f"{catalog_name}.{schema_name}.{table_name}")

In [0]:
# 4. Initialize Engine and Run
ws = WorkspaceClient()
engine = DQEngine(ws)

In [0]:
# This returns: (Passed Rows, Failed/Quarantined Rows)
valid_and_quarantine_df = engine.apply_checks(table_df,checks_config)
display(valid_and_quarantine_df)

In [0]:
# apply quality checks
valid_df, quarantine_df = engine.apply_checks_and_split(table_df, checks_config)

# save results
engine.save_results_in_table(
    output_df=valid_df,
    quarantine_df=quarantine_df,
    output_config=OutputConfig(f"{catalog_name}.{schema_name}.{table_name}_output", mode="overwrite"),
    quarantine_config=OutputConfig(f"{catalog_name}.{schema_name}.{table_name}_quarantine", mode="overwrite")
)

In [0]:
display(spark.table(f"{catalog_name}.{schema_name}.{table_name}_output"))
display(spark.table(f"{catalog_name}.{schema_name}.{table_name}_quarantine"))

## config - check by metadata

In [0]:
checks_config = []
for row in dqx_mapped_config_df.collect():
    import json
    args = {}
    if row['arguments']:
        for k, v in row['arguments'].items():
            try:
                args[k] = json.loads(v)
            except Exception:
                args[k] = v
    check_dict = {
        "name": row['name'],
        "column": row['column'],
        "function": row['function'],
        "criticality": row['criticality'],
        "arguments": args
    }
    checks_config.append({"check": check_dict})

In [0]:
for i in (checks_config):
    print(i)

In [0]:
# This returns: (Passed Rows, Failed/Quarantined Rows)
df = engine.apply_checks_by_metadata(table_df,checks_config)
display(df)

In [0]:
# end-to-end quality checking flow
engine.apply_checks_by_metadata_and_save_in_table(
    input_config=InputConfig(f"{catalog_name}.{schema_name}.{table_name}"),
    checks=checks_config,
    output_config=OutputConfig(f"{catalog_name}.{schema_name}.{table_name}_output", mode="overwrite"),
    quarantine_config=OutputConfig(f"{catalog_name}.{schema_name}.{table_name}_output", mode="overwrite")
)